In [1]:
import polars as pl

df = pl.read_parquet("./data/results.parquet")
df

date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
str,str,str,i64,i64,str,str,str,bool
"""1872-11-30""","""Scotland""","""England""",0,0,"""Friendly""","""Glasgow""","""Scotland""",false
"""1873-03-08""","""England""","""Scotland""",4,2,"""Friendly""","""London""","""England""",false
"""1874-03-07""","""Scotland""","""England""",2,1,"""Friendly""","""Glasgow""","""Scotland""",false
"""1875-03-06""","""England""","""Scotland""",2,2,"""Friendly""","""London""","""England""",false
"""1876-03-04""","""Scotland""","""England""",3,0,"""Friendly""","""Glasgow""","""Scotland""",false
…,…,…,…,…,…,…,…,…
"""2026-07-11""","""Argentina""","""Switzerland""",3,1,"""FIFA World Cup""","""Kansas City""","""United States""",true
"""2026-07-14""","""France""","""Spain""",0,2,"""FIFA World Cup""","""Arlington""","""United States""",true
"""2026-07-15""","""England""","""Argentina""",1,2,"""FIFA World Cup""","""Atlanta""","""United States""",true


In [ ]:
# Teams that play more than one match on the same day.
# Stack home/away teams into one column, group by (date, team), count matches.
same_day_teams = (
    pl.concat([
        df.select("date", pl.col("home_team").alias("team")),
        df.select("date", pl.col("away_team").alias("team")),
    ])
    .group_by("date", "team")
    .len()  # column named "len"
    .filter(pl.col("len") > 1)
    .sort("date", "team")
)

same_day_teams

In [12]:
team_names =  pl.concat([
    df.select("home_team").rename({"home_team": "team"}),
    df.select("away_team").rename({"away_team": "team"}),
]).unique().sort("team")

team_names.filter(
    pl.col("team").str.contains("Congo")
)

team
str
"""Congo"""
"""DR Congo"""


In [15]:
import json

with open("./data/team_names.json", "w") as f:
    json.dump(team_names.to_series().to_list(), f, indent=4)